**TASK1-a**

***Explain the PCY algorithm in your own words***

* PCY is an enhanced version of the Apriori algorithm that uses hashing to smartly reduce the number of candidate pairs considered in the second pass. During the first pass, besides counting individual items, it also hashes all pairs into a hash table. After the first pass, it uses the hash table to identify buckets that are likely to contain frequent pairs (based on their counts). In the second pass, only pairs of frequent items that hash to these "popular" buckets are counted as candidates. This filtering step makes PCY more efficient than Apriori by avoiding the counting of many pairs that are unlikely to be frequent.

**TASK1-b**

* Write a Python function to compute the support for each item and each
pair of items. 

In [3]:
from itertools import combinations

def compute_supports(baskets):

    supports = {}
    
    # Count individual items
    for basket in baskets:
        for item in basket:
            if item in supports:
                supports[item] += 1
            else:
                supports[item] = 1
    
    # Count pairs of items
    for basket in baskets:
        # Generate all possible pairs in the basket
        for pair in combinations(basket, 2):
            # Use frozenset to handle unordered pairs (since {a,b} == {b,a})
            frozen_pair = frozenset(pair)
            if frozen_pair in supports:
                supports[frozen_pair] += 1
            else:
                supports[frozen_pair] = 1
    
    return supports

In [4]:
baskets = [
    {'apple', 'banana', 'orange', 'grape'},
    {'banana', 'orange', 'grape', 'strawberry'},
    {'orange', 'grape', 'strawberry', 'pineapple'},
    {'grape', 'strawberry', 'pineapple', 'watermelon'},
    {'apple', 'banana', 'strawberry', 'pineapple'},
    {'banana', 'orange', 'pineapple', 'watermelon'},
    {'orange', 'grape', 'watermelon', 'peach'},
    {'apple', 'grape', 'strawberry', 'peach'},
    {'apple', 'banana', 'pineapple', 'peach'},
    {'banana', 'strawberry', 'pineapple', 'watermelon'},
    {'orange', 'strawberry', 'watermelon', 'peach'},
    {'grape', 'pineapple', 'watermelon', 'peach'}
]

supports = compute_supports(baskets)

# Print individual item supports
print("Individual item supports:")
for item, count in supports.items():
    if isinstance(item, str):
        print(f"{item}: {count}")

# Print pair supports
print("\nPair supports:")
for pair, count in supports.items():
    if isinstance(pair, frozenset):
        print(f"{set(pair)}: {count}")

Individual item supports:
orange: 6
grape: 7
banana: 6
apple: 4
strawberry: 7
pineapple: 7
watermelon: 6
peach: 5

Pair supports:
{'orange', 'grape'}: 4
{'orange', 'banana'}: 3
{'orange', 'apple'}: 1
{'grape', 'banana'}: 2
{'grape', 'apple'}: 2
{'banana', 'apple'}: 3
{'orange', 'strawberry'}: 3
{'strawberry', 'grape'}: 4
{'strawberry', 'banana'}: 3
{'orange', 'pineapple'}: 2
{'strawberry', 'pineapple'}: 4
{'pineapple', 'grape'}: 3
{'strawberry', 'watermelon'}: 3
{'pineapple', 'watermelon'}: 4
{'grape', 'watermelon'}: 3
{'strawberry', 'apple'}: 2
{'pineapple', 'banana'}: 4
{'pineapple', 'apple'}: 2
{'orange', 'watermelon'}: 3
{'banana', 'watermelon'}: 2
{'orange', 'peach'}: 2
{'peach', 'grape'}: 3
{'peach', 'watermelon'}: 3
{'strawberry', 'peach'}: 2
{'peach', 'apple'}: 2
{'peach', 'pineapple'}: 2
{'peach', 'banana'}: 1


**TASK1-f**

* Implement Python functions to verify your results for (c), (d), and (e).

In [5]:
# Item to numerical mapping
item_to_num = {
    'apple': 1,
    'banana': 2,
    'orange': 3,
    'grape': 4, 
    'strawberry': 5,
    'pineapple': 6,
    'watermelon': 7,
    'peach': 8
}

# Given baskets (using numerical values)
baskets = [
    {1, 2, 3, 4},
    {2, 3, 4, 5},
    {3, 4, 5, 6},
    {4, 5, 6, 7},
    {1, 2, 5, 6},
    {2, 3, 6, 7},
    {3, 4, 7, 8},
    {1, 4, 5, 8},
    {1, 2, 6, 8},
    {2, 5, 6, 7},
    {3, 5, 7, 8},
    {4, 6, 7, 8}
]

# Hash function: h(i,j) = (i * j) mod 13
def hash_pair(i, j):
    return (i * j) % 13

# Support threshold
s = 4

# (c): List all pairs and their buckets 
def generate_all_pairs():
    items = sorted(item_to_num.values())
    all_pairs = list(combinations(items, 2))
    pair_buckets = {}
    for pair in all_pairs:
        i, j = pair
        bucket = hash_pair(i, j)
        pair_buckets[pair] = bucket
    return pair_buckets

# (d): Identify frequent buckets 
def find_frequent_buckets():
    bucket_counts = {b: 0 for b in range(13)}  # Initialize all buckets (0-12)
    
    for basket in baskets:
        # Generate all pairs in the basket
        basket_pairs = combinations(basket, 2)
        for pair in basket_pairs:
            i, j = pair
            bucket = hash_pair(i, j)
            bucket_counts[bucket] += 1
    
    # Frequent buckets (count >= s)
    frequent_buckets = {b for b, count in bucket_counts.items() if count >= s}
    return bucket_counts, frequent_buckets

# Part (e): Determine pairs counted in the second pass
def get_second_pass_pairs(frequent_buckets):
    frequent_items = set(item_to_num.values())  # All items are frequent (support >=4)
    candidate_pairs = set()
    
    for basket in baskets:
        basket_pairs = combinations(basket, 2)
        for pair in basket_pairs:
            i, j = pair
            if i in frequent_items and j in frequent_items:
                bucket = hash_pair(i, j)
                if bucket in frequent_buckets:
                    candidate_pairs.add(frozenset(pair))  # Use frozenset to avoid duplicates
    
    return candidate_pairs

In [6]:
# (c): All pairs and their buckets
pair_buckets = generate_all_pairs()
print("===== Part (c): All Pairs and Their Buckets =====")
for pair, bucket in pair_buckets.items():
    print(f"Pair {pair}: Bucket {bucket}")

===== Part (c): All Pairs and Their Buckets =====
Pair (1, 2): Bucket 2
Pair (1, 3): Bucket 3
Pair (1, 4): Bucket 4
Pair (1, 5): Bucket 5
Pair (1, 6): Bucket 6
Pair (1, 7): Bucket 7
Pair (1, 8): Bucket 8
Pair (2, 3): Bucket 6
Pair (2, 4): Bucket 8
Pair (2, 5): Bucket 10
Pair (2, 6): Bucket 12
Pair (2, 7): Bucket 1
Pair (2, 8): Bucket 3
Pair (3, 4): Bucket 12
Pair (3, 5): Bucket 2
Pair (3, 6): Bucket 5
Pair (3, 7): Bucket 8
Pair (3, 8): Bucket 11
Pair (4, 5): Bucket 7
Pair (4, 6): Bucket 11
Pair (4, 7): Bucket 2
Pair (4, 8): Bucket 6
Pair (5, 6): Bucket 4
Pair (5, 7): Bucket 9
Pair (5, 8): Bucket 1
Pair (6, 7): Bucket 3
Pair (6, 8): Bucket 9
Pair (7, 8): Bucket 4


In [7]:
# (d): Frequent buckets
bucket_counts, frequent_buckets = find_frequent_buckets()
print("\n===== Part (d): Frequent Buckets (Count >= 4) =====")
print("Bucket counts:", bucket_counts)
print("Frequent buckets:", frequent_buckets)


===== Part (d): Frequent Buckets (Count >= 4) =====
Bucket counts: {0: 0, 1: 4, 2: 9, 3: 6, 4: 9, 5: 4, 6: 8, 7: 4, 8: 7, 9: 5, 10: 3, 11: 5, 12: 8}
Frequent buckets: {1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12}


In [8]:
# (e): Pairs counted in the second pass
second_pass_pairs = get_second_pass_pairs(frequent_buckets)
print("\n===== Part (e): Pairs Counted in Second Pass =====")
print("Number of candidate pairs:", len(second_pass_pairs))
print("Candidate pairs:")
for pair in second_pass_pairs:
    print(set(pair))


===== Part (e): Pairs Counted in Second Pass =====
Number of candidate pairs: 26
Candidate pairs:
{8, 2}
{1, 4}
{4, 6}
{2, 3}
{2, 6}
{4, 5}
{8, 7}
{3, 4}
{2, 4}
{5, 6}
{8, 1}
{3, 5}
{1, 6}
{8, 5}
{1, 3}
{4, 7}
{3, 7}
{6, 7}
{3, 6}
{8, 3}
{1, 5}
{8, 6}
{2, 7}
{5, 7}
{1, 2}
{8, 4}


**TASK1-g**
* Calculate the maximum possible number of distinct item pairs.

*Number of Pairs = [n x (n-1)] / 2*

Number of Pairs = [8 x (8-1)] / 2 

Number of Pairs = 28

**TASK1-h**

* *Discuss bucket choice. Explain why the number of buckets should be chosen carefully in the PCY algorithm. Discuss the trade-offs between too few and too many buckets with respect to accuracy and memory efficiency: *

* Best Practice: Use a prime number of buckets close to the expected number of frequent pairs.

* PCY’s Advantage: Even with some false positives, it reduces the candidate pairs significantly compared to Apriori.

* Memory vs. Accuracy Trade-off:

    * *Fewer buckets → Faster but less precise.*

    * *More buckets → Slower but more accurate.*

By carefully choosing the bucket count, PCY achieves a balance between efficiency and accuracy, making it superior to Apriori for large datasets.